# RAG : L'assistant virtuel de l'Hôtel Le Belvédère
## Partie 0 - Le nouveau brief de Marc


Quelques jours après notre démo, ce mail arrive dans la boîte du projet.

>De : Marc Belvédère
>
>Objet : Votre assistant, on continue !
>
>Bonjour,
>
>Je repense encore à ce que vous m'avez montré l'autre jour. Franchement, bravo. Julie, à la réception, a arrêté d'inventer des réponses quand un client lui pose une colle sur les animaux ou le spa, elle demande à votre assistant et hop. Elle a même bien ri quand mon neveu a voulu le piéger avec une histoire de paiement en Bitcoin et qu'il a refusé poliment de répondre. Ça, ça m'a rassuré : il ne dit pas n'importe quoi.
>
>Mais soyons sérieux deux minutes. Là c'était une démo sur votre ordinateur. Moi, je veux quelque chose de professionnel, quelque chose que je puisse mettre entre les mains de mon équipe avant la haute saison. Alors je vous demande de faire le nécessaire, et je compte sur vous pour bien faire les choses.
>
>Trois choses me chiffonnent.
>
>D'abord, votre truc, à chaque fois qu'on le relance il rumine un bon moment avant d'être prêt. Mes réceptionnistes, quand il y a la queue au comptoir, ils n'ont pas ce temps-là. Je veux que ce soit disponible tout de suite, à la seconde où on l'ouvre.
>
>Ensuite, et c'est le plus important : avant de laisser Julie s'en servir devant un vrai client, je veux être certain qu'il ne se trompe pas. Pas un "ça a l'air de marcher", non. Je veux une preuve, quelque chose de carré, avec des chiffres, un truc que je puisse poser sur la table devant mon associé et ma banque pour dire "regardez, c'est fiable, voici comment on l'a vérifié".
>
>Et enfin, soyons honnêtes, personne chez moi ne va aller taper dans vos lignes de code. Il me faut une vraie page, sur laquelle on écrit sa question et la réponse s'affiche, comme quand je discute sur mon téléphone. Simple, propre, que n'importe qui à la réception sache s'en servir sans formation.

>Faites ce qu'il faut. Le budget suit, mais je veux du sérieux.

>Bien à vous,

>Marc

> PS : Vous trouverez ci-joint un document supplémentaire à rajouter à votre truc, j'imagine que c'est facile pour vous, n'est-ce pas ?


### Import des packages

In [ ]:
import io

import chromadb
import requests
from chromadb.utils import embedding_functions
from pypdf import PdfReader


In [ ]:
def extraire_rubriques(pdf_bytes, source):
    """Une page de PDF = une rubrique. Titre = 1re ligne, footer retiré."""
    reader = PdfReader(io.BytesIO(pdf_bytes))
    rubriques = []
    for i, page in enumerate(reader.pages):
        lignes = [l.strip() for l in page.extract_text().split("\n") if l.strip()]
        # le footer est la ligne contenant "Documentation interne"
        lignes = [l for l in lignes if "Documentation interne" not in l]
        titre = lignes[0]
        corps = " ".join(lignes[1:])
        rubriques.append({
            "id": f"{source.replace('.pdf', '')}__{i}",
            "titre": titre,
            # on encode titre + corps (le titre résume le sujet, il aide la recherche)
            "document": f"## {titre}\n\n{corps}",
            "source": source,
        })
    return rubriques

In [ ]:
rubriques = []
for fichier in FICHIERS:
    r = requests.get(BASE_URL + fichier, timeout=30)
    r.raise_for_status()
    rubriques.extend(extraire_rubriques(r.content, fichier))

print(f"{len(rubriques)} rubriques extraites")
print(f"{rubriques}")

15 rubriques extraites
[{'id': 'activites_et_evenements__0', 'titre': 'Activités et alentours', 'document': "## Activités et alentours\n\nL'hôtel loue des vélos électriques à l'accueil pour 20 euros la demi-journée. Un ponton privé permet de pratiquer le paddle et le kayak gratuitement pour les clients, de juin à septembre. Le sentier de randonnée du Semnoz est accessible en 20 minutes de marche depuis l'hôtel. Le marché d'Annecy se tient le jeudi, le vendredi et le dimanche matin dans la vieille ville, à environ 10 minutes à pied. Les canaux et la vieille ville d'Annecy sont également accessibles à pied en 10 minutes. Une croisière touristique sur le lac part du ponton municipal, à 5 minutes de l'hôtel. Les thermes de Menthon-Saint-Bernard sont à 15 minutes en voiture. La plage de l'Impérial se trouve à 8 minutes à pied. Le golf d'Annecy est accessible en 10 minutes en voiture. En hiver, les stations de ski de La Clusaz et du Grand-Bornand sont à environ 30 minutes en voiture.", 'sour

### Modèle d'embedding MULTILINGUE

Le modèle utilisé par chromadb par défaut ne parle pas français, les embeddings seront donc mauvais, on doit le modifier

docs :https://docs.trychroma.com/docs/embeddings/embedding-functions

In [ ]:
fonction_embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

/home/onyxia/work/RAG-Hotel/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5182.72it/s]


### On génère une base persistante plutôt qu'un process on-memory



In [ ]:
client = chromadb.PersistentClient(path="./chroma_hotel")
collection = client.get_or_create_collection(
    name="hotel_collection",
    embedding_function=fonction_embedding,
    metadata={"hnsw:space": "cosine"},
)

### Ajout des méta-données pour filtrer

In [ ]:
collection.add(
    ids=[r["id"] for r in rubriques],
    documents=[r["document"] for r in rubriques],
    metadatas=[{"source": r["source"], "titre": r["titre"]} for r in rubriques],
)

print(f"{collection.count()} documents dans la collection")

15 documents dans la collection


In [ ]:
# Recherche simple
resultats = collection.query(query_texts=["Est-ce qu'il y a une piscine?"], n_results=1)
for doc, meta in zip(resultats["documents"][0], resultats["metadatas"][0]):
    print(meta["titre"], "|", doc[:60])



Piscine | ## Piscine

L'hôtel dispose d'une piscine extérieure chauffé


In [ ]:
collection.query(query_texts=["Est-ce qu'il y a une pataugeoire ?"], n_results=2)

{'ids': [['familles_animaux_accessibilite__2', 'informations_pratiques__3']],
 'embeddings': None,
 'documents': [["## Accessibilité PMR\n\nL'hôtel dispose de 4 chambres adaptées aux personnes à mobilité réduite, toutes situées au rez-de-chaussée, avec douche à l'italienne et barres d'appui. Un ascenseur accessible dessert tous les étages. Des places de parking réservées PMR sont disponibles devant l'entrée principale. Le restaurant et le spa sont accessibles en fauteuil roulant. Les chiens guides d'aveugle sont acceptés sans supplément, y compris dans les espaces habituellement interdits aux animaux.",
   "## Blanchisserie et services annexes\n\nUn service de blanchisserie et de pressing est proposé, avec collecte avant 9h et remise le soir même, sauf le dimanche où le service n'est pas assuré. Un service de repassage est disponible sur demande. Un cireur de chaussures automatique est à disposition à la réception. Chaque chambre est équipée d'un coffre-fort gratuit, suffisamment grand

In [ ]:
def search(question: str, k: int=2) -> list[dict]:
    """Récupère les k rubriques les plus proches d'une question.

    Encapsule la recherche vectorielle de Chroma et renvoie le résultat
    dans un format neutre, indépendant du store, que le reste du pipeline
    (génération, benchmark) peut consommer tel quel.

    Args:
        question: La question de l'utilisateur, en langage naturel.
        k: Nombre de rubriques à retourner. Defaults to 2.

    Returns:
        list[dict]: Les k rubriques les plus proches, triées du plus proche
            au moins proche. Chaque dict contient les clés :

            - ``titre`` (str): Le titre de la rubrique.
            - ``source`` (str): Le fichier PDF d'origine.
            - ``texte`` (str): Le contenu de la rubrique (titre + corps).
            - ``distance`` (float): La distance cosine à la question
              (plus petite = plus proche).
    """
    reponse = collection.query(query_texts=[question], n_results=k)

    rubriques = []
    for doc, meta, dist in zip(
        reponse["documents"][0],
        reponse["metadatas"][0],
        reponse["distances"][0],
    ):
        rubriques.append({
            "titre": meta["titre"],
            "source": meta["source"],
            "texte": doc,
            "distance": dist,
        })
    return rubriques

In [ ]:
questions_test = [
    "Est-ce qu'il y a une piscine ?",           # match une seule rubrique
    "Acceptez-vous les animaux ?",              # match une seule rubrique
    "piscine pour les enfants",                 # doit ramener Piscine ET Familles
    "Quels sont les tarifs des chambres ?",     # match une seule rubrique
]

for q in questions_test:
    print(f"\nQ: {q}")
    for r in search(q, k=2):
        print(f"   {r['distance']:.3f} | {r['titre']}")


Q: Est-ce qu'il y a une piscine ?
   0.430 | Piscine
   0.670 | Familles et enfants

Q: Acceptez-vous les animaux ?
   0.563 | Animaux domestiques
   0.877 | Accessibilité PMR

Q: piscine pour les enfants
   0.347 | Piscine
   0.453 | Familles et enfants

Q: Quels sont les tarifs des chambres ?
   0.352 | Chambres et tarifs
   0.498 | Spa et bien-être


### Build prompt & system prompt

In [ ]:
SYSTEM_PROMPT = (
    "Tu es l'assistant de l'Hôtel Le Belvédère, tu aides le personnel à "
    "répondre aux clients. Réponds UNIQUEMENT à partir du contexte fourni "
    "ci-dessous, de façon concise et en français. Si l'information ne figure "
    "pas dans le contexte, dis-le clairement plutôt que d'inventer."
)

def build_prompt(question: str, rubriques: list[dict]) -> list[dict]:
    """Assemble le prompt de génération au format chat (system + user).

    Cette fonction ne dépend d'aucun modèle : c'est le même prompt qui
    servira pour tous les SLM du benchmark. Seule la génération (morceau 2)
    est spécifique au modèle.

    Args:
        question (str): La question de l'utilisateur.
        rubriques (list[dict]): Les rubriques renvoyées par ``search()``.

    Returns:
        list[dict]: Les messages au format ``[{"role", "content"}, ...]``.
    """
    contexte = "\n\n".join(r["texte"] for r in rubriques)
    user = f"Contexte:\n{contexte}\n\nQuestion: {question}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]

### Charger le modèle

In [ ]:
# pip install transformers torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Chargé UNE SEULE FOIS (jamais dans la boucle de génération)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
)

def generate(question: str, k: int = 3) -> str:
    """Chaîne complète : retrieval puis génération.

    Args:
        question (str): La question de l'utilisateur.
        k (int, optional): Nombre de rubriques à récupérer. Defaults to 3.

    Returns:
        str: La réponse générée par le modèle.
    """
    rubriques = search(question, k=k)
    messages = build_prompt(question, rubriques)

    # Chaque modèle a SON format de chat : on passe par apply_chat_template
    texte = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(texte, return_tensors="pt").to("cpu")

    sortie = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,   # greedy : déterministe, indispensable pour un benchmark
    )
    # On ne décode que les tokens NOUVEAUX (on retire le prompt d'entrée)
    nouveaux = sortie[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(nouveaux, skip_special_tokens=True).strip()

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 7687.21it/s]


In [ ]:
for q in ["Est-ce qu'il y a une piscine pour enfants ?",
          "Acceptez-vous les animaux ?",
          "L'hôtel a-t-il une salle de sport ?"]:  # celle-ci est hors-scope
    print(f"\nQ: {q}")
    print(f"R: {generate(q)}")


Q: Est-ce qu'il y a une piscine pour enfants ?
R: Oui, il existe une piscine pour enfants avec une pataugeoire séparée.

Q: Acceptez-vous les animaux ?
R: Oui, les animaux domestiques de moins de 10 kg sont acceptés dans l'hôtel, moyennant un supplément de 15 euros par nuit et par animal.

Q: L'hôtel a-t-il une salle de sport ?
R: Non, l'hôtel n'a pas de salle de sport.


Qwen s'en sort très bien, et la troisième réponse mérite qu'on s'y arrête parce qu'elle est plutôt subtile

Sur les deux premières, rien à redire : français correct, ancrées dans le contexte, et surtout la réponse sur les animaux est précise (elle reprend le seuil de 10 kg et le supplément de 15 euros, donc elle n'oublie pas la condition).C'est bien le comportement attendu.

La troisième, la question hors-scope, est intéressante. La bonne nouvelle : Qwen n'a pas halluciné. Il n'a pas inventé une salle de sport avec des horaires imaginaires, ce qui était le vrai risque. À ce titre, il passe le test.

Mais regardons la formulation exacte : « Non, l'hôtel n'a pas de salle de sport. » Il affirme une absence comme un fait. Or ce que dit vraiment la documentation, c'est... rien. Elle ne mentionne pas de salle de sport, ce qui n'est pas la même chose que « il n'y en a pas ». 

Le modèle a transformé « ce n'est pas dans mon contexte » en « ça n'existe pas ». C'est un raisonnement par absence de preuve, et strictement, ce n'est pas parfaitement ancré. La réponse idéale était plutôt « cette information n'est pas dans la documentation »

### Et si on variait les modèles ?

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Le roster : SLM francophones + un témoin anglocentré (TinyLlama) pour le contraste
MODELES = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "HuggingFaceTB/SmolLM3-3B",
    "microsoft/Phi-4-mini-instruct",
    "croissantllm/CroissantLLMChat-v0.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    #"TinyLlama/TinyLlama-1.1B-Chat-v1.0",   # témoin faible en français
    # Gemma, SmolLM3, Phi-4-mini
]

def charger(nom: str):
    """Charge un modèle et son tokenizer sur CPU."""
    tok = AutoTokenizer.from_pretrained(nom)
    mod = AutoModelForCausalLM.from_pretrained(nom, torch_dtype="auto")
    return mod, tok

def liberer(model, tokenizer):
    """Libère la RAM avant de passer au modèle suivant."""
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
import os
import json
import time
import pandas as pd
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0
eval_df = pd.read_csv("../questions/eval_questions.csv")
CHEMIN = "resultats_benchmark.jsonl"      # JSONL : robuste aux virgules et retours à la ligne


def detecter_langue(txt: str) -> str:
    try:
        return detect(txt)
    except Exception:
        return "inconnu"


def stop_tokens(tokenizer) -> list[int]:
    """Token ids qui doivent arrêter la génération pour ce modèle.

    Combine l'eos standard et les tokens de fin de tour des formats chat
    courants, pour que chaque modèle s'arrête au bon endroit sans hack.
    """
    ids = set()
    if tokenizer.eos_token_id is not None:
        ids.add(tokenizer.eos_token_id)
    for tok in ["<|im_end|>", "<|eot_id|>", "<end_of_turn>", "</s>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            ids.add(tid)
    return list(ids)


def generate_with(model, tokenizer, question: str, k: int = 3) -> tuple[str, float, int]:
    """Chaîne retrieval + génération pour un modèle donné, avec mesure de latence.

    Args:
        model: Le modèle de génération chargé.
        tokenizer: Le tokenizer associé.
        question (str): La question de l'utilisateur.
        k (int, optional): Nombre de rubriques récupérées. Defaults to 3.

    Returns:
        tuple[str, float, int]: (réponse, latence en secondes, nb de tokens générés).
    """
    rubriques = search(question, k=k)
    messages = build_prompt(question, rubriques)
    texte = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(texte, return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    sortie = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=stop_tokens(tokenizer),
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    latence = time.perf_counter() - t0

    nouveaux = sortie[0][inputs["input_ids"].shape[1]:]
    reponse = tokenizer.decode(nouveaux, skip_special_tokens=True).strip()
    return reponse, latence, len(nouveaux)


def save(rows: dict):
    """Ajoute une ligne au fichier JSONL (un objet JSON par ligne)."""
    with open(CHEMIN, "a", encoding="utf-8") as f:
        f.write(json.dumps(rows, ensure_ascii=False) + "\n")


# 1. REPRISE : on recharge ce qui est déjà fait
if os.path.exists(CHEMIN):
    deja_fait = pd.read_json(CHEMIN, lines=True)
    faits = set(zip(deja_fait["modele"], deja_fait["id"]))
    print(f"Reprise : {len(faits)} réponses déjà présentes.")
else:
    faits = set()


# 2. BOUCLE avec écriture incrémentale
for nom in MODELES:
    ids_attendus = set(eval_df["id"])
    ids_faits = {qid for (m, qid) in faits if m == nom}
    if ids_attendus <= ids_faits:
        print(f"{nom} déjà complet, on saute.")
        continue

    print(f"Chargement de {nom}...")
    model, tokenizer = charger(nom)
    try:
        for _, ligne in eval_df.iterrows():
            if (nom, ligne["id"]) in faits:
                continue
            try:
                reponse, latence, n_tokens = generate_with(
                    model, tokenizer, ligne["question"]
                )
                save({
                    "modele": nom,
                    "id": int(ligne["id"]),
                    "type": ligne["type"],
                    "question": ligne["question"],
                    "reponse_attendue": ligne["reponse_attendue"],
                    "reponse_modele": reponse,
                    "langue": detecter_langue(reponse),
                    "latence_s": latence,
                    "tokens_s": n_tokens / latence if latence > 0 else 0,
                })
                faits.add((nom, ligne["id"]))
            except Exception as e:
                print(f"  ERREUR sur {nom} / q{ligne['id']} : {e}")
    finally:
        liberer(model, tokenizer)      # libéré même si la boucle explose
    print(f"  {nom} terminé.")


resultats_df = pd.read_json(CHEMIN, lines=True)
print(f"\n{len(resultats_df)} réponses au total dans {CHEMIN}.")

Reprise : 75 réponses déjà présentes.
Qwen/Qwen2.5-1.5B-Instruct déjà complet, on saute.
Qwen/Qwen2.5-0.5B-Instruct déjà complet, on saute.
Chargement de Qwen/Qwen2.5-3B-Instruct...


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 4199.90it/s]


KeyboardInterrupt: 

In [ ]:
resultats_df = pd.read_csv(CHEMIN, on_bad_lines="warn")   # saute les lignes malformées en prévenant
print(f"{len(resultats_df)} lignes lues")
print(resultats_df["modele"].value_counts())

64 lignes lues
modele
Qwen/Qwen2.5-1.5B-Instruct            22
Qwen/Qwen2.5-0.5B-Instruct            21
croissantllm/CroissantLLMChat-v0.1    21
Name: count, dtype: int64


/tmp/ipykernel_277890/2346220791.py:1: ParserWarning: Skipping line 45: expected 9 fields, saw 17

  resultats_df = pd.read_csv(CHEMIN, on_bad_lines="warn")   # saute les lignes malformées en prévenant


In [ ]:
df = pd.read_json("resultats_benchmark.jsonl", lines=True)
df["m"] = df["modele"].apply(lambda n: n.split("/")[-1])

print("Réponses par modèle (doit être 22 chacun) :")
print(df.groupby("m").size().to_string())

print("\n=== Langue + vitesse + longueur ===")
df["len_car"] = df["reponse_modele"].str.len()
synth = df.groupby("m").agg(
    pct_francais=("langue", lambda s: round(100 * (s == "fr").mean(), 1)),
    latence_med_s=("latence_s", "median"),
    tokens_s_moy=("tokens_s", "mean"),
    len_moy_car=("len_car", "mean"),
).round(1)
print(synth.to_string())

Réponses par modèle (doit être 22 chacun) :
m
CroissantLLMChat-v0.1    22
Qwen2.5-0.5B-Instruct    22
Qwen2.5-1.5B-Instruct    22

=== Langue + vitesse + longueur ===
                       pct_francais  latence_med_s  tokens_s_moy  len_moy_car
m                                                                            
CroissantLLMChat-v0.1         100.0           19.6           1.2        160.9
Qwen2.5-0.5B-Instruct         100.0           26.0           1.2        119.6
Qwen2.5-1.5B-Instruct         100.0           30.8           1.0        107.2


In [ ]:
df

,modele,id,type,question,reponse_attendue,reponse_modele,langue,latence_s,tokens_s,m,len_car
0,Qwen/Qwen2.5-1.5B-Instruct,1,nuance,Le petit-déjeuner est-il inclus dans le tarif ...,"Non, il coûte 18 euros par personne en supplém...","Oui, le petit-déjeuner est inclus dans le tari...",fr,9.472732,2.005757,Qwen2.5-1.5B-Instruct,56
1,Qwen/Qwen2.5-1.5B-Instruct,2,numérique,Quel est le prix d'une chambre Deluxe Vue Lac ?,De 210 à 260 euros la nuit selon la saison.,La chambre Deluxe Vue Lac coûte de 210 à 260 e...,fr,17.689032,1.469837,Qwen2.5-1.5B-Instruct,76
2,Qwen/Qwen2.5-1.5B-Instruct,3,factuel,Jusqu'à quand puis-je annuler gratuitement ma ...,Jusqu'à 48h avant la date d'arrivée. Passé ce ...,Jusqu'à 48 heures avant la date d'arrivée.,fr,11.594389,1.379978,Qwen2.5-1.5B-Instruct,42
3,Qwen/Qwen2.5-1.5B-Instruct,4,factuel,Les animaux sont-ils acceptés à l'hôtel ?,"Oui, ceux de moins de 10 kg, avec un supplémen...","Oui, les animaux domestiques de moins de 10 kg...",fr,27.401555,1.532760,Qwen2.5-1.5B-Instruct,143
4,Qwen/Qwen2.5-1.5B-Instruct,5,limite,Puis-je venir avec mon chien de 20 kg ?,"Non, seuls les animaux de moins de 10 kg sont ...","Non, les animaux de plus de 10 kg ne sont pas ...",fr,19.101123,1.204118,Qwen2.5-1.5B-Instruct,68
...,...,...,...,...,...,...,...,...,...,...,...
61,croissantllm/CroissantLLMChat-v0.1,18,nuance,Un chien guide d'aveugle est-il accepté au res...,"Oui, les chiens guides sont acceptés sans supp...","Oui, un chien guide d'aveugle est accepté au r...",fr,10.392733,1.347095,CroissantLLMChat-v0.1,56
62,croissantllm/CroissantLLMChat-v0.1,19,factuel,Y a-t-il une navette pour la gare ?,"Oui, une navette gratuite dessert la gare d'An...","Non, il n'y a pas de navette pour la gare.",fr,12.114086,1.155679,CroissantLLMChat-v0.1,42
63,croissantllm/CroissantLLMChat-v0.1,20,nuance,Le service de blanchisserie fonctionne-t-il le...,"Non, le service n'est pas assuré le dimanche. ...","oui, le service de blanchisserie fonctionne le...",fr,15.831194,0.821164,CroissantLLMChat-v0.1,56
64,croissantllm/CroissantLLMChat-v0.1,21,hors-scope,L'hôtel dispose-t-il d'une salle de sport ?,L'information n'est pas mentionnée dans la doc...,"Oui, l'hôtel dispose d'une salle de sport avec...",fr,12.638637,1.661572,CroissantLLMChat-v0.1,100
